# Tutorial 4-1: Auditing for Bias in Traditional ML
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
As we discussed in class, machine learning systems can echo or even amplify existing societal biases. In this tutorial, we will take a 'traditional' approach to auditing a model for fairness. We will explore:
1. **Performance Disparity:** Measuring how accuracy varies across different demographic groups.
2. **False Positive Rates:** Understanding the 'cost' of a wrong prediction for different individuals.
3. **The Importance of Intersectionality:** Analyzing how multiple attributes (e.g., Race + Gender) combine to create unique bias patterns.

## 1. Setup and Data Loading
We will use the **Adult Census Income dataset**. Our goal is to predict if an individual earns more than $50,000/year based on attributes like education, age, and occupation. While this seems like a standard task, the data contains historical wage gaps that our model might learn to enforce.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Load the dataset (simplified version for tutorial purposes)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age', 'workclass', 'fnlwgt', 'education', 'edu_num', 'marital', 'occupation', 
           'relationship', 'race', 'sex', 'cap_gain', 'cap_loss', 'hours_per_week', 'country', 'income']
df = pd.read_csv(url, names=columns, skipinitialspace=True)

# Preprocessing
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)
X = pd.get_dummies(df.drop('income', axis=1))
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Standard Model (Biased)
model_standard = LogisticRegression(max_iter=1000)
model_standard.fit(X_train_scaled, y_train)
y_pred_std = model_standard.predict(X_test_scaled)
print(f"Overall Model Accuracy: {accuracy_score(y_test, y_pred_std):.4f}")


## 2. Auditing for Performance Disparity
A high overall accuracy can hide significant failures for specific subgroups. Let's calculate the accuracy specifically for 'Male' vs 'Female' groups to see if the model works equally well for everyone.

In [ ]:
# Extract indices for gender groups in the test set
test_indices = X_test.index
sex_male = df.loc[test_indices, 'sex'] == 'Male'
sex_female = df.loc[test_indices, 'sex'] == 'Female'

acc_male = accuracy_score(y_test[sex_male], y_pred_std[sex_male])
acc_female = accuracy_score(y_test[sex_female], y_pred_std[sex_female])

plt.figure(figsize=(8, 5))
sns.barplot(x=['Male Accuracy', 'Female Accuracy'], y=[acc_male, acc_female])
plt.title("Accuracy Disparity by Gender")
plt.ylim(0, 1)
plt.show()

print(f"Male Accuracy: {acc_male:.4f}")
print(f"Female Accuracy: {acc_female:.4f}")

## 3. Investigating False Positive Rates (FPR)
In this context, a False Positive means predicting someone earns >$50k when they actually earn less. In other scenarios (like loan approvals or criminal sentencing), a False Positive for one group and a False Negative for another can lead to severe 'unequal impacts'.

In [ ]:
def get_fpr(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return fp / (fp + tn)

fpr_male = get_fpr(y_test[sex_male], y_pred_std[sex_male])
fpr_female = get_fpr(y_test[sex_female], y_pred_std[sex_female])

print(f"False Positive Rate (Male): {fpr_male:.4f}")
print(f"False Positive Rate (Female): {fpr_female:.4f}")
print(f"Ratio: {fpr_male / fpr_female:.2f}x higher for men")

## Summary
Our traditional model successfully 'learned' the wage patterns in the data, but in doing so, it also learned to be more accurate for one gender than the other. 
1. **Disparity:** Even a simple model can have unequal error rates across groups.
2. **Harm:** False Positive disparities can lead to 'privileged' groups receiving more favorable automated decisions.
3. **Solution:** Awareness is the first step. As data scientists, auditing our models is as important as training them.